## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

## 2. Load Dataset

In [ ]:
# Load dataset

df = pd.read_csv('dataset/air_quality/air_quality.csv')

# Show first rows

df.head()

## 3. Dataset Information

In [ ]:
print(df.shape)
print(df.columns)
# Check missing values

df.isnull().sum()


## 4. Convert Time Column

In [ ]:
# Convert datetime column

df['ts_utc'] = pd.to_datetime(df['ts_utc'])

# Sort by time

df = df.sort_values('ts_utc')

# Reset index

df.reset_index(drop=True, inplace=True)

## 5. Select Target Column

In [ ]:
# Select AQI column

target_col = 'aqi'

series = df[target_col]

## 6. Visualize Time Series

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df['ts_utc'], series)
plt.title('AQI Time Series')
plt.xlabel('Time')
plt.ylabel('AQI')
plt.show()

## 7. Train / Validation / Test Split (70/20/10)

In [ ]:
n = len(df)

train_size = int(n * 0.7)
val_size   = int(n * 0.2)

test_size  = n - train_size - val_size

train = series[:train_size]
val   = series[train_size:train_size+val_size]
test  = series[train_size+val_size:]

print('Train size:', len(train))
print('Validation size:', len(val))
print('Test size:', len(test))

## 8. Plot Dataset Split

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(train.index, train, label='Train')
plt.plot(val.index, val, label='Validation')
plt.plot(test.index, test, label='Test')

plt.legend()
plt.title('Dataset Split')
plt.show()

## 9. Evaluation Metrics Function

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print(f'MAE  : {mae:.6f}')
    print(f'MSE  : {mse:.6f}')
    print(f'RMSE : {rmse:.6f}')
    print(f'R2   : {r2:.6f}')

    return mae, mse, rmse, r2

## 10. Naive Forecast Model

In [ ]:
# Use previous value as prediction

naive_pred = []

history = list(train)

for t in range(len(test)):
    yhat = history[-1]
    naive_pred.append(yhat)
    history.append(test.iloc[t])

## Evaluate Naive Model

In [ ]:
evaluate(test, naive_pred)

## Plot Naive Forecast

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(test.values, label='True')
plt.plot(naive_pred, label='Naive Prediction')

plt.legend()
plt.title('Naive Forecast')
plt.show()

## 11. Moving Average Model

In [ ]:
window_size = 5

moving_avg_pred = []

history = list(train)

for t in range(len(test)):
    yhat = np.mean(history[-window_size:])
    moving_avg_pred.append(yhat)
    history.append(test.iloc[t])

## Evaluate Moving Average

In [ ]:
evaluate(test, moving_avg_pred)

## Plot Moving Average Forecast

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(test.values, label='True')
plt.plot(moving_avg_pred, label='Moving Average Prediction')

plt.legend()
plt.title('Moving Average Forecast')
plt.show()

## 12. Check Stationarity for ARIMA/SARIMA
Augmented Dickey-Fuller (ADF) Test

In [ ]:
result = adfuller(train)

print('ADF Statistic:', result[0])
print('p-value:', result[1])

Interpretation
p-value < 0.05 → stationary
p-value > 0.05 → non-stationary

## 13. Differencing

In [ ]:
train_diff = train.diff().dropna()

## Check Stationarity Again

In [ ]:
result_diff = adfuller(train_diff)

print('ADF Statistic:', result_diff[0])
print('p-value:', result_diff[1])

## 14. ARIMA Model
ARIMA(p,d,q)

Example:

p = 2
d = 1
q = 2

In [ ]:
arima_model = ARIMA(train, order=(2,0,2))

arima_fit = arima_model.fit()

print(arima_fit.summary())

## Forecast Using ARIMA

In [ ]:
arima_pred = arima_fit.forecast(steps=len(test))

## Evaluate ARIMA

In [ ]:
evaluate(test, arima_pred)

## Plot ARIMA Forecast

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(test.values, label='True')
plt.plot(arima_pred, label='ARIMA Prediction')

plt.legend()
plt.title('ARIMA Forecast')
plt.show()

## 15. SARIMA Model
SARIMA(p,d,q)(P,D,Q,s)

Example:

Seasonal period s = 24
Suitable for hourly data

In [ ]:
sarima_model = SARIMAX(
    train,
    order=(2,0,2),
    seasonal_order=(1,1,1,24)
)

sarima_fit = sarima_model.fit(disp=False)

print(sarima_fit.summary())

## Forecast Using SARIMA

In [ ]:
sarima_pred = sarima_fit.forecast(steps=len(test))

## Evaluate SARIMA

In [ ]:
evaluate(test, sarima_pred)

## Plot SARIMA Forecast

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(test.values, label='True')
plt.plot(sarima_pred, label='SARIMA Prediction')

plt.legend()
plt.title('SARIMA Forecast')
plt.show()

## 16. Compare All Models

In [ ]:
results = []

# Naive
mae, mse, rmse, r2 = evaluate(test, naive_pred)
results.append(['Naive', mae, mse, rmse, r2])

# Moving Average
mae, mse, rmse, r2 = evaluate(test, moving_avg_pred)
results.append(['Moving Average', mae, mse, rmse, r2])

# ARIMA
mae, mse, rmse, r2 = evaluate(test, arima_pred)
results.append(['ARIMA', mae, mse, rmse, r2])

# SARIMA
mae, mse, rmse, r2 = evaluate(test, sarima_pred)
results.append(['SARIMA', mae, mse, rmse, r2])

## Create Comparison Table

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2']
)

results_df

## 17. Visualization Comparison

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(results_df['Model'], results_df['RMSE'])

plt.title('RMSE Comparison')
plt.ylabel('RMSE')
plt.show()